# Imports

In [1]:
import yfinance as yf
import pandas as pd
import os

# Extract

In [2]:
SP500_UNIVERSE = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "BRK-B",
    "JPM", "JNJ", "V", "PG", "UNH", "HD", "MA", "DIS", "PYPL", "BAC",
    "NFLX", "ADBE"
]

In [3]:
def ingest_ticker(ticker: str, start: str, end: str) -> pd.DataFrame:
    data = yf.download(ticker, start=start, end=end, auto_adjust=True)
    data = data.reset_index()
    data.columns = [c[0].lower().replace(" ", "_") for c in data.columns]
    data["ticker"] = ticker
    return data

In [4]:
prices = pd.DataFrame()
print(f"Tipo de prices_df antes del bucle: {type(prices)}")
for ticker in SP500_UNIVERSE:
    df = ingest_ticker(ticker, "2017-01-01", "2024-12-31")
    prices = pd.concat([prices,df])

Tipo de prices_df antes del bucle: <class 'pandas.DataFrame'>


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

# Transform

In [5]:
prices['date'] = pd.to_datetime(prices['date'])
prices['week'] = prices['date'].dt.isocalendar().week
prices['year'] = prices['date'].dt.isocalendar().year
prices['month'] = prices['date'].dt.month


In [6]:
prices[['date','month','year','week']]

,date,month,year,week
0,2017-01-03,1,2017,1
1,2017-01-04,1,2017,1
2,2017-01-05,1,2017,1
3,2017-01-06,1,2017,1
4,2017-01-09,1,2017,2
...,...,...,...,...
2006,2024-12-23,12,2024,52
2007,2024-12-24,12,2024,52
2008,2024-12-26,12,2024,52
2009,2024-12-27,12,2024,52


In [12]:
prices["ticker"] = prices["ticker"].astype(str)
prices["year"] = prices["year"].astype(int)

# Load

In [15]:
path = "../data/prices_clean/"

for (ticker, year), group in prices.groupby(["ticker", "year"]):
    dir_path = f"{path}/ticker={ticker}/year={year}"
    os.makedirs(dir_path, exist_ok=True)
    
    # 🔥 quitar columnas de partición
    group = group.drop(columns=["ticker", "year"])
    
    group.to_parquet(f"{dir_path}/data.parquet", index=False)

In [0]:



# def ingest_ticker(ticker: str, start: str, end: str) -> pd.DataFrame:
#     data = yf.download(ticker, start=start, end=end, auto_adjust=True)
#     data = data.reset_index()
#     data.columns = [c[0].lower().replace(" ", "_") for c in data.columns]
#     data["ticker"] = ticker
#     return data

# def transform(prices_pdf: pd.DataFrame):
#     prices_sdf = spark.createDataFrame(prices_pdf)
#     return (prices_sdf.withColumn("year",year(col("date")))
#                 .withColumn("month",month(col("date")))
#                 .withColumn("date",to_date(col("date"))))
# def load(prices_sdf: pd.DataFrame, path: str):
#     (prices_sdf.write
#                 .option("overwriteSchema", "True")
#                 .format("delta")
#                 .mode("overwrite")
#                 .partitionBy("ticker","year")
#                 .save(path))
    
# prices_df = pd.DataFrame()
# print(f"Tipo de prices_df antes del bucle: {type(prices_df)}")
# for ticker in SP500_UNIVERSE:
#     df = ingest_ticker(ticker, "2021-01-01", "2024-12-31")
#     prices_df = pd.concat([prices_df,df])
# transformed = transform(prices_df)
# load(transformed, "/Volumes/market-mood/processed/prices_clean/")



In [0]:
# SP500_UNIVERSE = [
#     "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "BRK-B",
#     "JPM", "JNJ", "V", "PG", "UNH", "HD", "MA", "DIS", "PYPL", "BAC",
#     "NFLX", "ADBE"
# ]


# def ingest_ticker(ticker: str, start: str, end: str) -> pd.DataFrame:
#     data = yf.download(ticker, start=start, end=end, auto_adjust=True)
#     data = data.reset_index()
#     data.columns = [c[0].lower().replace(" ", "_") for c in data.columns]
#     data["ticker"] = ticker
#     return data

    

# def extract(tickers: list, start: str, end: str) -> pd.DataFrame:
#     prices_df = pd.DataFrame()
#     print(f"Tipo de prices_df antes del bucle: {type(prices_df)}")
#     for ticker in tickers:
#         df = ingest_ticker(ticker, start, end)
#         prices_df = pd.concat([prices_df,df])
#     return prices_df

# def transform(prices_pdf: pd.DataFrame):
#     prices_sdf = spark.createDataFrame(prices_pdf)
#     return (prices_sdf.withColumn("year",year(col("date")))
#                 .withColumn("month",month(col("date")))
#                 .withColumn("date",to_date(col("date"))))



# def load(prices_sdf: DataFrame, path: str):
#     (prices_sdf.write
#                 .option("overwriteSchema", "True")
#                 .format("delta")
#                 .mode("overwrite")
#                 .partitionBy("ticker","year")
#                 .save(path))

# # Ejecución principal
# if __name__ == "__main__":
#     raw = extract(SP500_UNIVERSE, "2021-01-01", "2024-12-31")
#     transformed = transform(raw)
#     load(transformed, "/Volumes/market-mood/raw/prices")